In [ ]:
import os
# os.listdir('/home/kickd/Downloads/gmx_demo/')

In [ ]:
# Example data

# For building the graph
kegg_cache_path = '/home/kickd/Downloads/gmx_demo/'
# For connecting the graph to the genetic data
gff_path = '/home/kickd/Downloads/gmx_demo/ncbi_gmx_refseq_gff/ncbi_dataset/data/GCF_000004515.6/genomic.gff'
# Genetic data
vcf_path = '/home/kickd/Downloads/gmx_demo/SoySNP50K_genotyping_Gillman_GWAS.vcf'
hmp_path = '/home/kickd/Downloads/gmx_demo/SoySNP50K_genotyping_Gillman_GWAS.hmp.txt'

# Phenotypic data
phno_path = '/home/kickd/Downloads/gmx_demo/phno.csv'

cache_path = '/home/kickd/Downloads/gmx_demo/DemoNb/'
lightning_log_dir = cache_path+"lightning"
exp_name = [e for e in cache_path.split('/') if e != ''][-1]

In [ ]:
import numpy  as np
import pandas as pd


import einops

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import sparsevnn
import sparsevnn.util
from   sparsevnn.core import \
    SparseLinearCustom, \
    dist_scale_function, \
    info_list_to_layer_list, \
    SparseVNN, \
    MarkerDataset, \
    VNNHelper, \
    structured_layer_info, \
    plDNN_general

    
# sparsevnn.util.read_vcf
# sparsevnn.util.vcf_table_to_matrix
# sparsevnn.util.hmp_table_to_matrix


# Hyperparameter Tuning ----
import os # needed for checking history (saved by lightning) 

## Logging with Pytorch Lightning ====
import lightning.pytorch as pl
from   lightning.pytorch.loggers import CSVLogger # used to save the history of each trial (used by ax)

## Adaptive Experimentation Platform ====
from ax.service.ax_client import AxClient, ObjectiveProperties
# from ax.utils.notebook.plotting import init_notebook_plotting, render


torch.set_float32_matmul_precision('medium')

In [ ]:
def ensure_dir_path_exists(dir_path = '../ext_data' # Directory path to check
                          ):
    "Iteratively check for and create directories to store output. Ideally this would just be os.mkdirs() but that function is not available in this version of python"
    import os
    
    for i in range(2, len(dir_path.split('/'))+1):
        path_part = '/'.join(dir_path.split('/')[0:i])
        if not os.path.exists(path_part):
            os.mkdir(path_part)

In [ ]:
ensure_dir_path_exists(dir_path = cache_path)

## Load Marker Data 
This workflow now uses vcf or hmp files to load marker data. hapmaps are prefered in that they are _meaningfully_ faster to process (32.2s -> 1.0s for a 32553 snp x 365 genotype table). 

In [ ]:
# vcf = sparsevnn.util.read_vcf(vcf_path)
# acgt= sparsevnn.util.vcf_table_to_matrix(vcf=vcf)

hmp = pd.read_table(hmp_path)
hmp = hmp.sort_values(['chrom', 'pos']).reset_index(drop=True)
acgt= sparsevnn.util.hmp_table_to_matrix(hmp)


# Currently acgt is in a "rotation" (order of dimenisons) that was convenient for its creation. 
# We'll rotate it from snps, genotype, prob. -> genotype, prob., snps which is the order we need for the model.
acgt = einops.rearrange(acgt, 's g p -> g p s')

In [ ]:
acgt_taxa = [e for e in list(hmp) 
    if e not in ['rs#', 'alleles', 'chrom', 'pos', 'strand', 'assembly#', 'center', 'protLSID', 'assayLSID', 'panelLSID', 'QCcode']]

In [ ]:
acgt_loci = hmp.loc[:, ['chrom', 'pos']]

## Load Phenotypes and metadata

The key columns here are the target phenotype(s) (`Yield_Mg_ha`) and columns to match this up with genomic and/or enviromental data (`Env`, `Year`, `Hybrid`). The columns for the enviroment aren't strictly necessary but could be useful if you want to predict residual to an enviromental mean instead of a trait directly. 

I'll also call your attention to the `Phno_Idx`, `Env_Idx`, and `Geno_Idx` columns. These provide a way to go from the row in a phenotype table (the row number in the original dataset is `Unnamed: 0` redundant with `Phno_Idx`) to a row in an enviroment or genotype table. 

For model training instead of providing a table with all the phenotypes and one with all the predictors, we instead provide a linker table for each type of predictiors (genotype, soil, weather, treatment group, etc.). This lets use a _deduplicated_ version of the table which can save a ton of memory. 

During training we get the index of an observation
| Y |
|---|
| # | 
| # | 
| # |

use that G index to get a G index
| Y | G |
|---|---|
| 1 | 1 |
| 2 | 1 |
| 3 | 2 |

and that G index retrieves the right genotype.
| snp1 | snp2 | ... | snpm |
|---|---|---|---|
| # | # | # | # |
| # | # | # | # |




In [ ]:
# Taxa, y1, y2 ...
phno = pd.read_csv(phno_path)
phno.head()

,Taxa,ProteinDry,OilDry
0,4J105_3_4,40.24,21.57
1,4J105_3_4,41.31,21.16
2,4J105_3_4,40.05,20.95
3,4J105_3_4,41.24,21.16
4,4J105_3_4,40.94,21.25


Filter Taxa based on availability

In [ ]:
phno_taxa   = list(set(phno.Taxa.tolist()))
shared_taxa = sorted([e for e in phno_taxa if e in acgt_taxa])

In [ ]:
phno = phno.loc[(phno.Taxa.isin(shared_taxa)), ].reset_index(drop = True)

### Outcome variable(s)
The trait or traits to be predicted we'll turn into an array. We'll return to it later for a few more changes. 

For cases where we have a single could drop the last axis and have shape of (obs,) but if we want multiple outputs we\'ll need a 2 dimensional array (a rank 2 tensor) so keeping this dimension is important.

In [ ]:
y = np.array(phno.drop(columns='Taxa'))
y_names = list(phno.drop(columns='Taxa'))
print(f'The output array is of shape {y.shape}.')

The output array is of shape (3305, 2).


### Prepare Lookup tables

We want to use a deduplicated tensor of genomic data. To accomplish this we're going to store a lookup to match phenotype to genotype.


|Phno_Idx | Geno_Idx | Is_Phno |
|---------|----------|---------|
|Row of phenotype | Row of Genotype | First instance of Genotype in Phenotype Table |

In [ ]:
unique_geno = phno.loc[:, ['Taxa']].drop_duplicates().reset_index(drop=True).reset_index().rename(columns={'index':'Geno_Idx'})
unique_geno.head()

,Geno_Idx,Taxa
0,0,4J105_3_4
1,1,5M20_2_5_2
2,2,AdYdTr2014_1
3,3,AdYdTr2014_10
4,4,AdYdTr2014_2


In [ ]:
unique_geno = phno.loc[:, ['Taxa']].drop_duplicates(
).reset_index().rename(columns={'index':'Is_Phno'}
).sort_values('Taxa'                                # These are sorted to mirror `shared_taxa`
).reset_index().rename(columns={'index':'Geno_Idx'}
)
unique_geno.head()

,Geno_Idx,Is_Phno,Taxa
0,0,0,4J105_3_4
1,1,9,5M20_2_5_2
2,2,18,AdYdTr2014_1
3,3,27,AdYdTr2014_10
4,4,36,AdYdTr2014_2


In [ ]:
obs_geno_lookup = phno.loc[:, ['Taxa']
                           ].reset_index().rename(columns={'index':'Phno_Idx'}
                           ).merge(unique_geno, how='outer').drop(columns = ['Taxa'])
obs_geno_lookup

,Phno_Idx,Geno_Idx,Is_Phno
0,0,0,0
1,1,0,0
2,2,0,0
3,3,0,0
4,4,0,0
...,...,...,...
3300,3300,364,3296
3301,3301,364,3296
3302,3302,364,3296
3303,3303,364,3296


## Build or Load Graph Structure

These functions retrieve a specific KEGG functional hierarchy and parse that file into a table of sources and targets.

In [ ]:
# catalog = sparsevnn.util._get_available_catalog(species = 'gmx')
inp = sparsevnn.util._get_json(species = 'gmx', catalog_num = '00001', cache = True, cache_dir = kegg_cache_path)
inp = sparsevnn.util._peel(inp=inp)
cxn = sparsevnn.util._connections_from_peeled_json(inp, max_iter = 1000, print_queue_len = False)
cxn = pd.DataFrame(cxn, columns=['src', 'tgt'])
cxn.head()

,src,tgt
0,gmx00001,09100 Metabolism
1,gmx00001,09120 Genetic Information Processing
2,gmx00001,09130 Environmental Information Processing
3,gmx00001,09140 Cellular Processes
4,gmx00001,09150 Organismal Systems


### Match Graph Inputs to Gene Models (Parsing GFF annotation file)

We need a way to convert the ids used by KEGG to and from those used by NCBI. Thankfully KEGG provides access to a lookup through their api. We can build a reference using `_get_kegg2ncbi`

In [ ]:
kegg2ncbi = sparsevnn.util._get_kegg2ncbi(species = 'gmx', cache = True, cache_dir = kegg_cache_path)
ncbi2kegg = {kegg2ncbi[k]:k for k in kegg2ncbi}

For this example I have downloaded a genome annotation from NCBI: https://www.ncbi.nlm.nih.gov/datasets/genome/GCF_000004515.6/ . 

In [ ]:
gff = sparsevnn.util._read_gene_annotation_table(filepath = gff_path)
gff = sparsevnn.util._gene_annotation_table_expand_attributes(gff)
# project chromosome over rows
gff = gff.loc[(gff.chromosome.notna()), ['seqid', 'chromosome']].merge( gff.drop(columns=['chromosome']) )
# select columns
gff = gff.loc[(gff.type == 'gene'), ['chromosome', 'start', 'end', 'ID', 'Dbxref']]
# Drop any without a known chromosome.
gff = gff.loc[(gff.chromosome != 'Unknown')]
gff.head()

,chromosome,start,end,ID,Dbxref
1,1,51942,53819,gene-LOC100808170,GeneID:100808170
8,1,53914,74178,gene-LOC102661143,GeneID:102661143
588,1,74455,75201,gene-LOC121174904,GeneID:121174904
592,1,76301,77728,gene-LOC106794262,GeneID:106794262
632,1,94227,100053,gene-LOC100781438,GeneID:100781438


Now we can use `kegg2ncbi` to match up the input nodes with those that have a ncbi-geneid

In [ ]:
species = 'gmx'

#TODO cleave the species code and ncbi-geneid so we don't have to add in `species` here.

nodes = list(set(cxn.src.tolist()+cxn.tgt.tolist()))
# since the lamda deals with missing entries below we don't need to filter the inputs
# gene_nodes = [e for e in nodes if re.match('\d+.*', e)]
gene_nodes = nodes

gene_nodes = pd.DataFrame(zip(
    gene_nodes,
    [f"{species}:{e.split(' ')[0]}" for e in gene_nodes]), columns=['cxn', 'kegg'])

gene_nodes['ncbi']= [# lambda to deal with missing values
    (lambda x: kegg2ncbi[x] if x in kegg2ncbi.keys() else '')(e) 
    for e in gene_nodes.kegg.tolist()]

# Transform Dbxref and join
gff['ncbi'] = [f"ncbi-geneid:{e.split(':')[-1]}" for e in gff['Dbxref'].tolist()]

gene_nodes_gff = gene_nodes.merge(gff, how='inner')
gene_nodes_gff = gene_nodes_gff.sort_values(['chromosome', 'start', 'end']).reset_index(drop=True)

print('\n'.join(
    ['Grouping\t| Count', 
     '--------\t| -----'
     ]+[f'{i}\t| {j}' for i,j in zip(
         ['All KEGG Nodes', 'GFF Nodes', 'Intersection'],
         [e.shape[0] for e in [gene_nodes, gff, gene_nodes_gff]]
         )]))

gene_nodes_gff.head()

Grouping	| Count
--------	| -----
All KEGG Nodes	| 11737
GFF Nodes	| 53910
Intersection	| 11122


,cxn,kegg,ncbi,chromosome,start,end,ID,Dbxref
0,100780723 secretory carrier-associated membran...,gmx:100780723,ncbi-geneid:100780723,1,1005950,1011552,gene-LOC100780723,GeneID:100780723
1,100305353 SLTI248; DEAD-box RNA helicase\tK148...,gmx:100305353,ncbi-geneid:100305353,1,1013916,1018556,gene-SLTI248,GeneID:100305353
2,100813174 pectinesterase PPME1\tK01051 E3.1.1....,gmx:100813174,ncbi-geneid:100813174,1,10175387,10177214,gene-LOC100813174,GeneID:100813174
3,100816036 pectinesterase PPME1\tK01051 E3.1.1....,gmx:100816036,ncbi-geneid:100816036,1,10321136,10323014,gene-LOC100816036,GeneID:100816036
4,100306177 putative pectinesterase precursor\tK...,gmx:100306177,ncbi-geneid:100306177,1,10388090,10389975,gene-LOC100306177,GeneID:100306177


In [ ]:
# FIXME - As a test case I'm choosing a handful of genes that overlap in some of their input snps. 
# This overlap will occur most frequently when genes fall in between markers because then we're using values from the nearest two markers. 
# We want to make sure that the input matrix is not concatenated slices per gene. If it is, then we duplicate (potentially) many values. 

# Here are a few that should have overlapping values. 
_ = { 
    '100793480 uncharacterized protein LOC100793480\tK03039 PSMD13; 26S proteasome regulatory subunit N9': [34, 35],
    '100798066 vacuolar protein sorting-associated protein 25 isoform X1\tK12189 VPS25; ESCRT-II complex subunit VPS25': [34, 35],

    '100813174 pectinesterase PPME1\tK01051 E3.1.1.11; pectinesterase [EC:3.1.1.11]': [158, 159],
    '100816036 pectinesterase PPME1\tK01051 E3.1.1.11; pectinesterase [EC:3.1.1.11]': [158, 159],
    '100306177 putative pectinesterase precursor\tK01051 E3.1.1.11; pectinesterase [EC:3.1.1.11]': [158, 159],

    '100785374 transcription factor MYB53\tK09422 MYBP; transcription factor MYB, plant': [160, 161]
    }

gene_nodes_gff = gene_nodes_gff.loc[gene_nodes_gff.cxn.isin(_.keys()), ].reset_index(drop = True)

We now have a way to match the connections in the graph to positions in the genome by the chromosome, start, end fields. 

In [ ]:
# Filter the input nodes to only those that we have a gene model for. 

# This should be run multiple times until there are no nodes that are either
# 1. Leaves that have no snps
# 2. Branches without leaves that have no snps.

# Instead of using a while loop, we iterate over the number of nodes. 
print('Removing nodes without SNPs:')
for itr in range(len(list(set(cxn.tgt+cxn.src)))):
    # The input nodes will be targets but not sources
    tgt =  list(set(cxn.tgt))
    src =  list(set(cxn.src))
    
    inp_nodes = [e for e in tgt if e not in src]
    # here are the input nodes that we don't have a gene model for.
    inp_nodes_prune = [e for e in inp_nodes if e not in gene_nodes_gff.cxn.tolist()]
    print(f'{itr}: {len(inp_nodes_prune)} of {len(inp_nodes)} ({round(100*(len(inp_nodes_prune)/len(inp_nodes)), 3)} %)')
    cxn = cxn.loc[~(cxn.tgt.isin(inp_nodes_prune)), ]
    if inp_nodes_prune == []:
        break

cxn = cxn.reset_index(drop=True)

Removing nodes without SNPs:
0: 11591 of 11597 (99.948 %)
1: 115 of 121 (95.041 %)
2: 12 of 18 (66.667 %)
3: 1 of 7 (14.286 %)
4: 0 of 6 (0.0 %)


The connection data frame is now clean and ready to go. We're able to convert between representations of the graph with the `convert_connections` function (e.g. `sparsevnn.util.convert_connections(inp=cxn, to='dict', node_names=None)`). The connections can also be used to order the nodes from root to leaves.  

``` 
cxn_order_dict = sparsevnn.util.order_connections(inp = cxn, node_names=None)
sum([cxn_order_dict[e] for e in list(cxn_order_dict.keys())], [])

# ['gmx00001',
#  '09180 Brite Hierarchies',
#  '09100 Metabolism',
#  '09190 Not Included in Pathway or Brite',
#  '09120 Genetic Information Processing',
#  ...
#  '100779110 25.3 kDa vesicle transport protein\tK08517 SEC22; vesicle transport protein SEC22']

```

## Update Genotype data

In [ ]:
# confirm that shared_taxa and the geno_index have the same order 
assert False == (False in [i == j for i,j in zip(
    unique_geno.sort_values('Geno_Idx').reset_index(drop=True).Taxa.tolist(), 
    shared_taxa)])

In [ ]:
# Make sure the order of the taxa in acgt are as expected
# currently dims are taxa, nucleotide, length
print(acgt.shape)

taxa2idx = {k:v for v,k in enumerate(acgt_taxa)}
acgt = acgt[[taxa2idx[e] for e in shared_taxa], :, :]

Here is one way of linking snps to genes. Instead of finding snps within a given gene we look for the indices that are closest to or within.  

In [ ]:
gene_nodes_gff
acgt_loci

,chrom,pos
0,1,24952
1,1,26003
2,1,29671
3,1,37018
4,1,46373
...,...,...
32548,20,47862732
32549,20,47879138
32550,20,47881223
32551,20,47884469


In [ ]:
acgt_loci = acgt_loci.reset_index().rename(columns={'index':'acgt_l_idx'})
acgt_loci.head()

In [ ]:
# A  B  C  D  E  F    <- snps sampled
#    #######          <- geneic region
# 
# we return indexs for [A, B, C, D, E]
# 
# Whereas for gene:
# A  B  C  D  E  F    <- snps sampled
#      #              <- geneic region
# 
# we return indexs for [B, C] since there are not indexes within the gene
#
# and for gene:
# A  B  C  D  E  F    <- snps sampled
#                   # <- geneic region
# 
# we return indexs for [F] since there is no snp sampled at a higher position.


# make sure we're working with ints
acgt_loci.chrom = acgt_loci.chrom.astype(int)
acgt_loci.pos   = acgt_loci.pos.astype(int)

out = {}

for i in gene_nodes_gff.index:
    node, chromosome, start, end = gene_nodes_gff.loc[i, ['cxn', 'chromosome', 'start', 'end']].tolist()
    node, chromosome, start, end = str(node), int(chromosome), int(start), int(end)
    # node, chromosome, start, end

    chrom_mask = (acgt_loci.chrom == chromosome)

    res = []
    # index that's closest but below the gene
    res += acgt_loci.loc[
        (chrom_mask & (acgt_loci.pos < start)), 
        ['acgt_l_idx']].max().tolist()

    # indexes within gene
    res += acgt_loci.loc[
        (chrom_mask & 
        ((acgt_loci.pos >= start) &
        (acgt_loci.pos <= end))
        ), 
        'acgt_l_idx'].tolist()

    # index that's closest but above the gene
    res += acgt_loci.loc[
        (chrom_mask & (acgt_loci.pos > end)), 
        ['acgt_l_idx']].min().tolist()
    
    # drop nans
    res = [e for e in res if e == e]
    out = out | {node:res}


In [ ]:
# Do we actually need the full set of genome snp indices or can we drop some?
used_snps = sorted(list(set(sum([out[e] for e in out.keys()], []))))
print(f'Using {len(used_snps)} of {acgt.shape[-1]} SNPs ({round(100*(len(used_snps) / acgt.shape[-1]), 3)}%)')

In [ ]:
# Reduce the snp dim of acgt and then update all the references in out. 
idxorig2reduced = {k:i for i,k in enumerate(used_snps)}
acgt = acgt[:, :, used_snps]

out = {
    k:[idxorig2reduced[e] for e in v] 
    for k,v in 
    [(k, out[k]) for k in out.keys()]}

inp_node_idx_dict = out.copy()

In [ ]:
# inp_node_idx_dict

## Build model

In [ ]:
cxn_dict = sparsevnn.util.convert_connections(inp=cxn, to='dict', node_names=None)
# dependancy_order = sparsevnn.util.order_connections(inp = cxn, node_names=None)
# dependancy_order = sum([dependancy_order[e] for e in list(dependancy_order.keys())], [])

In [ ]:
def mk_vnnhelper(
        edge_dict,
        inp_tensor_lookup,
        num_nucleotides = 4, # this could also be 1 for major/minor allele. 
        params = {
            'default_out_nodes_inp'  : 1,
            'default_out_nodes_edge' : 1,
            'default_out_nodes_out'  : 1, #TODO set this based on the dimensions of y

            'default_drop_nodes_inp' : 0.0,
            'default_drop_nodes_edge': 0.0,
            'default_drop_nodes_out' : 0.0,

            'default_reps_nodes_inp' : 1,
            'default_reps_nodes_edge': 1,
            'default_reps_nodes_out' : 1,

            'default_decay_rate'     : 0
            }
            ):        
    # older code assumes that a graph dictionary will contain leaves as keys with [] children. 
    # to accomodate this behavior we're going to 
    # 1. check if there are any nodes that are not keys and
    # 2. if there are spike them in. 
    all_nodes = list(set(sum([[k]+edge_dict[k] for k in edge_dict.keys()], [])))
    absent_nodes = {e:[] for e in all_nodes if e not in edge_dict.keys()}
    if absent_nodes != {}:
        edge_dict = edge_dict | absent_nodes
    # Now we don't need to worry about which structure the connection dict has

    myvnn = VNNHelper(edge_dict = edge_dict)

    # We need to set attributes of the VNNHelper so the edges can be calculated.

    myvnn.set_node_props(
        key = 'inp', 
        node_val_zip = zip(
            myvnn.nodes_inp, 
            [len(inp_tensor_lookup[e])*num_nucleotides for e in myvnn.nodes_inp]
            ))

    myvnn.set_node_props(
        key = 'flatten', 
        node_val_zip = zip(myvnn.nodes_inp, [True for e in myvnn.nodes_inp]))

    for node_group in ['nodes_inp', 'nodes_edge', 'nodes_out']:
        for attr_type in ['out', 'drop', 'reps']:
            myvnn.set_node_props(
                key= attr_type, 
                node_val_zip=zip(
                    myvnn.__getattribute__(node_group),
                    [params[f'default_{attr_type}_{node_group}']   # repeat relevant default value
                    for e in myvnn.__getattribute__(node_group)]
                    # an example version of this is 
                    # myvnn.set_node_props(
                    #     key = 'reps', 
                    #     node_val_zip = zip(myvnn.nodes_inp, [default_reps_nodes_inp  for e in myvnn.nodes_inp]))
            ))


    # Scale node outputs by distance -----------------------------------------------
    dist = sparsevnn.core.vertex_from_end(
        edge_dict = myvnn.edge_dict,
        end =myvnn.dependancy_order[-1]
    )

    # overwrite node outputs with a size inversely proportional to distance from prediction node
    for query in list(dist.keys()):
        myvnn.node_props[query]['out'] = dist_scale_function(
            out = myvnn.node_props[query]['out'],
            dist = dist[query],
            decay_rate = params['default_decay_rate'])
        
    # Expand out node replicates ---------------------------------------------------
    nodes = [node for node in myvnn.dependancy_order if myvnn.node_props[node]['reps'] > 1]

    node_expansion_dict = {
        node: [node if i==0 else f'{node}_{i}' for i in range(myvnn.node_props[node]['reps'])]
        for node in nodes}
    #   current       1st          2nd (new)      3rd (new)
    # {'100798274': ['100798274', '100798274_1', '100798274_2'], ...

    # the keys don't change here. The values will be updated and then new k:v will be inserted
    myvnn.edge_dict = {k:[e if e not in node_expansion_dict.keys() 
        else node_expansion_dict[e][-1]
        for e in myvnn.edge_dict[k] ] for k in myvnn.edge_dict}

    # now insert connectsion to new nodes: A -> A_rep_1 -> A_rep_2
    for node in node_expansion_dict:
        for pair in zip(node_expansion_dict[node][1:], node_expansion_dict[node]):
            myvnn.edge_dict[pair[0]] = [pair[1]]

    # now add those new nodes
    # create a new node for all the nodes
    for node in node_expansion_dict:
        for new_node in node_expansion_dict[node][1:]:
            myvnn.node_props[new_node] = {k:myvnn.node_props[node][k] for k in myvnn.node_props[node] if k != 'inp'}


    new_vnn = VNNHelper(edge_dict= myvnn.edge_dict)
    new_vnn.node_props = myvnn.node_props
    myvnn = new_vnn

    # init edge node input size (propagate forward input/edge outpus)
    myvnn.calc_edge_inp()
    return myvnn


In [ ]:
params = {
    'default_out_nodes_inp'  : 1,
    'default_out_nodes_edge' : 1,
    'default_out_nodes_out'  : 2, #TODO set this based on the dimensions of y

    'default_drop_nodes_inp' : 0.0,
    'default_drop_nodes_edge': 0.0,
    'default_drop_nodes_out' : 0.0,

    'default_reps_nodes_inp' : 1,
    'default_reps_nodes_edge': 1,
    'default_reps_nodes_out' : 1,

    'default_decay_rate'     : 0
    }

myvnn = mk_vnnhelper(
        edge_dict = cxn_dict,
        num_nucleotides = 4, # this could also be 1 for major/minor allele. 
        inp_tensor_lookup = inp_node_idx_dict,
        params = params
            )

In [ ]:
# edge_dict = cxn_dict
# num_nucleotides = 4 # this could also be 1 for major/minor allele. 
# inp_tensor_lookup = inp_node_idx_dict 

# params = {
# 'default_out_nodes_inp'  : 1,
# 'default_out_nodes_edge' : 1,
# 'default_out_nodes_out'  : 2, #TODO set this based on the dimensions of y

# 'default_drop_nodes_inp' : 0.0,
# 'default_drop_nodes_edge': 0.0,
# 'default_drop_nodes_out' : 0.0,

# 'default_reps_nodes_inp' : 1,
# 'default_reps_nodes_edge': 1,
# 'default_reps_nodes_out' : 1,

# 'default_decay_rate'     : 0
# }

In [ ]:
# def mk_vnnhelper():        
#     # older code assumes that a graph dictionary will contain leaves as keys with [] children. 
#     # to accomodate this behavior we're going to 
#     # 1. check if there are any nodes that are not keys and
#     # 2. if there are spike them in. 
#     all_nodes = list(set(sum([[k]+edge_dict[k] for k in edge_dict.keys()], [])))
#     absent_nodes = {e:[] for e in all_nodes if e not in edge_dict.keys()}
#     if absent_nodes != {}:
#         edge_dict = edge_dict | absent_nodes
#     # Now we don't need to worry about which structure the connection dict has

#     myvnn = VNNHelper(edge_dict = edge_dict)

#     # We need to set attributes of the VNNHelper so the edges can be calculated.

#     # inp_node_idx_dict[ myvnn.nodes_inp[0] ]

#     # options should be controlled by node_props


#     myvnn.set_node_props(
#         key = 'inp', 
#         node_val_zip = zip(
#             myvnn.nodes_inp, 
#             [len(inp_node_idx_dict[e])*num_nucleotides for e in myvnn.nodes_inp]
#             ))

#     myvnn.set_node_props(
#         key = 'flatten', 
#         node_val_zip = zip(myvnn.nodes_inp, [True for e in myvnn.nodes_inp]))

#     for node_group in ['nodes_inp', 'nodes_edge', 'nodes_out']:
#         for attr_type in ['out', 'drop', 'reps']:
#             myvnn.set_node_props(
#                 key= attr_type, 
#                 node_val_zip=zip(
#                     myvnn.__getattribute__(node_group),
#                     [params[f'default_{attr_type}_{node_group}']   # repeat relevant default value
#                     for e in myvnn.__getattribute__(node_group)]
#                     # an example version of this is 
#                     # myvnn.set_node_props(
#                     #     key = 'reps', 
#                     #     node_val_zip = zip(myvnn.nodes_inp, [default_reps_nodes_inp  for e in myvnn.nodes_inp]))
#             ))


#     # Scale node outputs by distance -----------------------------------------------
#     dist = sparsevnn.core.vertex_from_end(
#         edge_dict = myvnn.edge_dict,
#         end =myvnn.dependancy_order[-1]
#     )

#     # overwrite node outputs with a size inversely proportional to distance from prediction node
#     for query in list(dist.keys()):
#         myvnn.node_props[query]['out'] = dist_scale_function(
#             out = myvnn.node_props[query]['out'],
#             dist = dist[query],
#             decay_rate = params['default_decay_rate'])
        
#     # Expand out node replicates ---------------------------------------------------
#     nodes = [node for node in myvnn.dependancy_order if myvnn.node_props[node]['reps'] > 1]

#     node_expansion_dict = {
#         node: [node if i==0 else f'{node}_{i}' for i in range(myvnn.node_props[node]['reps'])]
#         for node in nodes}
#     #   current       1st          2nd (new)      3rd (new)
#     # {'100798274': ['100798274', '100798274_1', '100798274_2'], ...

#     # the keys don't change here. The values will be updated and then new k:v will be inserted
#     myvnn.edge_dict = {k:[e if e not in node_expansion_dict.keys() 
#         else node_expansion_dict[e][-1]
#         for e in myvnn.edge_dict[k] ] for k in myvnn.edge_dict}

#     # now insert connectsion to new nodes: A -> A_rep_1 -> A_rep_2
#     for node in node_expansion_dict:
#         for pair in zip(node_expansion_dict[node][1:], node_expansion_dict[node]):
#             myvnn.edge_dict[pair[0]] = [pair[1]]

#     # now add those new nodes
#     # create a new node for all the nodes
#     for node in node_expansion_dict:
#         for new_node in node_expansion_dict[node][1:]:
#             myvnn.node_props[new_node] = {k:myvnn.node_props[node][k] for k in myvnn.node_props[node] if k != 'inp'}


#     new_vnn = VNNHelper(edge_dict= myvnn.edge_dict)
#     new_vnn.node_props = myvnn.node_props
#     myvnn = new_vnn

#     # init edge node input size (propagate forward input/edge outpus)
#     myvnn.calc_edge_inp()
#     return myvnn


In [ ]:
# start of vnn_factory_2 replacement
# dependancy_order = sparsevnn.util.order_connections(inp = myvnn.edge_dict, node_names=None)
dd = sparsevnn.core.mk_NodeGroups(edge_dict=myvnn.edge_dict, dependancy_order=myvnn.dependancy_order)
dd.keys()

In [ ]:
# I think the key change I need to make here is to allow for positions to be provided. 

M_list = [
    # sparsevnn.core.structured_layer_info(
    structured_layer_info(
    i = ii, 
    node_groups=dd, 
    node_props=myvnn.node_props, 
    edge_dict=myvnn.edge_dict, 
    as_sparse=True,
    inp_tensor_nucleotides= 4,
    # lambda to only provide the lookup for the 0th grouping (input level)
    inp_tensor_lookup = (lambda x: inp_node_idx_dict if x == 0 else None)(ii)
    )
    for ii in sorted(list(dd.keys()))]

[list(e.weight.shape) for e in M_list]


In [ ]:
print('\n'.join(
    ['Layer\tinp\tout\teye'
    ]+[f"{k}\t{len(dd[k]['inp'])}\t{len(dd[k]['out'])}\t{len(dd[k]['eye'])}" for k in dd]
    ))

In [ ]:
# What mode are we in? 
# Hyperparameter tuning?
# Model training?
# Inference?
# 


In [ ]:
# ## Settings ====
# # run_hyps = 32 
# # run_hyps_force = False # should we run more trials even if the target number has been reached?
# # max_hyps = 64

# # Run settings: 
# params_run = {
#     'batch_size': 256,
#     'max_epoch' : 16, #256,    
# }

# # data settings
# params_data = {
#     # 'y_var': 'Yield_Mg_ha',
#     'y_var': [
#         # Description quoted from competition data readme
#         'Yield_Mg_ha',     # Grain yield in Mg per ha at 15.5% grain moisture, using plot area without alley (Mg/ha).
#         # 'Pollen_DAP_days', # Number of days after planting that 50% of plants in the plot began shedding pollen.
#         # 'Silk_DAP_days',   # Number of days after planting that 50% of plants in the plot had visible silks.
#         # 'Plant_Height_cm', # Measured as the distance between the base of a plant and the ligule of the flag leaf (centimeter).
#         # 'Ear_Height_cm',   # Measured as the distance from the ground to the primary ear bearing node (centimeter).
#         # 'Grain_Moisture',  # Water content in grain at harvest (percentage).
#         # 'Twt_kg_m3'        # Shelled grain test weight (kg/m3), a measure of grain density.
#     ],

#     'y_resid': 'Env', # None, Env, Geno
#     'y_resid_strat': 'naive_mean', # None, naive_mean, filter_mean, ...
#     'holdout_parents': [
#         ## 2022 ====
#         'LH244',
#         ## 2021 ====
#         'PHZ51',
#         # 'PHP02',
#         # 'PHK76',
#         ## 2019 ====
#         # 'PHT69',
#         'LH195',
#         ## 2017 ====
#         # 'PHW52',
#         # 'PHN82',
#         ## 2016 ====
#         # 'DK3IIH6',
#         ## 2015 ====
#         # 'PHB47',
#         # 'LH82',
#         ## 2014 ====
#         # 'LH198',
#         # 'LH185',
#         # 'PB80',
#         # 'CG102',
#  ],    
# }


batch_size = 32
max_epoch  = 8

In [ ]:
#TODO set definition

_ = obs_geno_lookup.Geno_Idx.drop_duplicates().tolist().copy()

rng = np.random.default_rng(8923747)
rng.shuffle(_)

trn_cutoff = round(len(_) * 0.8)

train_idx = obs_geno_lookup.loc[(obs_geno_lookup.Geno_Idx.isin(_[0:trn_cutoff])), 'Phno_Idx'].tolist()
test_idx = obs_geno_lookup.loc[(obs_geno_lookup.Geno_Idx.isin(_[trn_cutoff:])), 'Phno_Idx'].tolist()



In [ ]:
# import sparsevnn.core
# 'info_list_to_layer_list' in dir(sparsevnn.core)

In [ ]:
acgt_tensor = torch.from_numpy(acgt
                  ).to(torch.float
                  ).swapaxes(1,2 # obs, nucleotide, length -> obs, length, nucleotide so that
                  ).reshape(acgt.shape[0], -1) # reshape will but the nucleotides right next to each other. This will make the gene lookup make sense.

In [ ]:
y = torch.from_numpy(y
                  ).to(torch.float
                  )

y_c = y[train_idx].mean(axis=0)
y_s = y[train_idx].std(axis=0)

y = (y - y_c)/y_s

In [ ]:
training_dataloader = DataLoader(
    MarkerDataset(
        lookup_obs = torch.from_numpy(np.array(train_idx)),
        G = acgt_tensor,
        y = y.to(torch.float32),
        lookup_geno = torch.from_numpy(obs_geno_lookup.to_numpy())
        ),
        batch_size = batch_size,
        shuffle = True 
)

validation_dataloader = DataLoader(
    MarkerDataset(
        lookup_obs = torch.from_numpy(np.array(test_idx)),
        G = acgt_tensor,
        y = y.to(torch.float32),
        lookup_geno = torch.from_numpy(obs_geno_lookup.to_numpy())
        ),
        batch_size = batch_size,
        shuffle = False 
)

# next(iter(training_dataloader))

In [ ]:
# here's the shape of the data
[e.shape for e in next(iter(training_dataloader))]

In [ ]:
obs_geno_lookup

In [ ]:
# layer_list = info_list_to_layer_list(M_list = M_list, nonlinearity = F.relu)
# model      = SparseVNN(layer_list = layer_list)
# VNN        = plDNN_general(model)

# optimizer = VNN.configure_optimizers()
# logger    = CSVLogger(lightning_log_dir, name=exp_name)
# logger.log_hyperparams(params={
#     'params': params,
#     # 'params_data': tmp_params
# })
# trainer = pl.Trainer(max_epochs=max_epoch, logger=logger)
# trainer.fit(model=VNN, train_dataloaders=training_dataloader, val_dataloaders=validation_dataloader)

## Hyperparameter tuning

Going to assume that these values are in the global scope. 

In [ ]:
def train_one_model(
    params = params,
    edge_dict = cxn_dict,
    num_nucleotides = 4, # this could also be 1 for major/minor allele. 
    inp_tensor_lookup = inp_node_idx_dict
    ):
    myvnn = mk_vnnhelper(
            edge_dict = edge_dict,
            num_nucleotides = num_nucleotides, # this could also be 1 for major/minor allele. 
            inp_tensor_lookup = inp_tensor_lookup,
            params = params
                )

    dd = sparsevnn.core.mk_NodeGroups(edge_dict=myvnn.edge_dict, dependancy_order=myvnn.dependancy_order)

    M_list = [
        structured_layer_info(
        i = ii, 
        node_groups=dd, 
        node_props=myvnn.node_props, 
        edge_dict=myvnn.edge_dict, 
        as_sparse=True,
        inp_tensor_nucleotides= num_nucleotides,
        # lambda to only provide the lookup for the 0th grouping (input level)
        inp_tensor_lookup = (lambda x: inp_tensor_lookup if x == 0 else None)(ii)
        )
        for ii in sorted(list(dd.keys()))]

    layer_list = info_list_to_layer_list(M_list = M_list, nonlinearity = F.relu)
    model      = SparseVNN(layer_list = layer_list)
    VNN        = plDNN_general(model)

    optimizer = VNN.configure_optimizers()
    logger    = CSVLogger(lightning_log_dir, name=exp_name)
    logger.log_hyperparams(params={
        'params': params,
        # 'params_data': tmp_params
    })
    trainer = pl.Trainer(max_epochs=max_epoch, logger=logger)
    trainer.fit(model=VNN, train_dataloaders=training_dataloader, val_dataloaders=validation_dataloader)

In [ ]:
# save out 
# VNN.mod

In [ ]:
def evaluate(parameterization):
    train_one_model(
        params = parameterization,
        edge_dict = cxn_dict,
        num_nucleotides = 4, # this could also be 1 for major/minor allele. 
        inp_tensor_lookup = inp_node_idx_dict
    )
    
    # if we were optimizing number of training epochs this would be an effective loss to use.
    # trainer.callback_metrics['train_loss']
    # float(trainer.callback_metrics['train_loss'])

    # To potentially _overtrain_ models and still let the selction be based on their best possible performance,
    # I'll use the lowest average error in an epoch
    log_path = lightning_log_dir+'/'+exp_name
    fls = os.listdir(log_path)
    nums = [int(e.split('_')[-1]) for e in fls] 

    M = pd.read_csv(log_path+f"/version_{max(nums)}/metrics.csv")
    M = M.loc[:, ['epoch', 'train_loss']].dropna()

    M = M.groupby('epoch').agg(
        train_loss = ('train_loss', 'mean'),
        train_loss_sd = ('train_loss', 'std'),
        ).reset_index()

    train_metric = M.train_loss.min()
    print(train_metric)
    return {"train_loss": (train_metric, 0.0)}

# evaluate(parameterization = params)

In [ ]:
## Settings ====
run_hyps = 2 
run_hyps_force = False # should we run more trials even if the target number has been reached?
max_hyps = 2

# Run settings: 
params_run = {
    'batch_size': 1,
    'max_epoch' : 20,    
}

# data settings
params_data = {
    'y_var': y_names,
    # I've removed parameters for what should be held out, if residuals should be predicted, etc. 
}

In [ ]:
batch_size = params_run['batch_size']
max_epoch  = params_run['max_epoch']

y_var = params_data['y_var']

In [ ]:
params_list = [    
    ## Output Size ====
    {
    'name': 'default_out_nodes_inp',
    'type': 'range',
    'bounds': [1, 8],
    'value_type': 'int',
    'log_scale': False
    },
    {
    'name': 'default_out_nodes_edge',
    'type': 'range',
    'bounds': [1, 32],
    'value_type': 'int',
    'log_scale': False
    },
    {
    'name': 'default_out_nodes_out',
    'type': 'fixed',
    'value': 1, #NOTE This will be overwritten below
    'value_type': 'int',
    'log_scale': False
    },
    ## Dropout ====
    {
    'name': 'default_drop_nodes_inp',
    'type': 'range',
    'bounds': [0.01, 0.99],
    'value_type': 'float',
    'log_scale': False
    },
    {
    'name': 'default_drop_nodes_edge',
    'type': 'range',
    'bounds': [0.01, 0.99],
    'value_type': 'float',
    'log_scale': False
    },
    {
    'name': 'default_drop_nodes_out',
    'type': 'range',
    'bounds': [0.01, 0.99],
    'value_type': 'float',
    'log_scale': False,
    'sort_values':True
    },
    ## Node Repeats ====
    {
    'name': 'default_reps_nodes_inp',
    'type': 'choice',
    'values': [1, 2, 3],
    'value_type': 'int',
    'is_ordered': True,
    'sort_values':True
    },
    {
    'name': 'default_reps_nodes_edge',
    'type': 'choice',
    'values': [1, 2, 3],
    'value_type': 'int',
    'is_ordered': True,
    'sort_values':True
    },
    {
    'name': 'default_reps_nodes_out',
    'type': 'choice',
    'values': [1, 2, 3],
    'value_type': 'int',
    'is_ordered': True,
    'sort_values':True
    },
    ## Node Output Size Scaling ====
    {
    'name': 'default_decay_rate',
    'type': 'choice',
    'values': [0+(0.1*i) for i in range(10)]+[1.+(1*i) for i in range(11)],
    'value_type': 'float',
    'is_ordered': True,
    'sort_values':True
    }
    ]

In [ ]:
# overwrite params_list's output with the size with the right output size. Don't allow the user to enter the wrong value. 
# This means we don't need to worry much about re-using these values. 
i = [i for i in range(len(params_list)) if params_list[i]['name'] == 'default_out_nodes_out'][0]
params_list[i]['value'] = y.shape[1]

In [ ]:
# lightning_log_dir = cache_path+"lightning"
# exp_name = [e for e in cache_path.split('/') if e != ''][-1]

In [ ]:
# save_prefix = [e for e in cache_path.split('/') if e != ''][-1]






In [ ]:
# use_gpu_num = 0

# device = "cuda" if torch.cuda.is_available() else "cpu"
# if use_gpu_num in [0, 1]: 
#     torch.cuda.set_device(use_gpu_num)
# print(f"Using {device} device")

In [ ]:
# def evaluate(parameterization):
#     myvnn, new_lookup_dict = vnn_factory_1(parsed_kegg_gene_entries = kegg_gene_entries, # <-- Note to self, naming differs from zma, gmx
#                                            params = parameterization, ACGT_gene_slice_list = ACGT_gene_slice_list)
#     M_list = vnn_factory_2(vnn_helper = myvnn, node_to_inp_num_dict = new_lookup_dict)
#     layer_list =  vnn_factory_3(M_list = M_list)
#     model = NeuralNetwork(layer_list = layer_list)
    
#     VNN = plDNN_general(model)  
#     # optimizer = VNN.configure_optimizers()
#     VNN.configure_optimizers()
#     logger = CSVLogger(lightning_log_dir, name=exp_name)
#     logger.log_hyperparams(params={
#         'params': parameterization
#     })

#     trainer = pl.Trainer(max_epochs=max_epoch, logger=logger)
#     trainer.fit(model=VNN, train_dataloaders=training_dataloader, val_dataloaders=validation_dataloader)


#     # if we were optimizing number of training epochs this would be an effective loss to use.
#     # trainer.callback_metrics['train_loss']
#     # float(trainer.callback_metrics['train_loss'])

#     # To potentially _overtrain_ models and still let the selction be based on their best possible performance,
#     # I'll use the lowest average error in an epoch
#     log_path = lightning_log_dir+'/'+exp_name
#     fls = os.listdir(log_path)
#     nums = [int(e.split('_')[-1]) for e in fls] 

#     M = pd.read_csv(log_path+f"/version_{max(nums)}/metrics.csv")
#     M = M.loc[:, ['epoch', 'train_loss']].dropna()

#     M = M.groupby('epoch').agg(
#         train_loss = ('train_loss', 'mean'),
#         train_loss_sd = ('train_loss', 'std'),
#         ).reset_index()

#     train_metric = M.train_loss.min()
#     print(train_metric)
#     return {"train_loss": (train_metric, 0.0)}


In [ ]:
# ax_client = AxClient()
# ax_client.create_experiment(
#     name=exp_name,
#     parameters=params_list,
#     objectives={"train_loss": ObjectiveProperties(minimize=True)}
# )
# ax_client.get_next_trial()


In [ ]:
## Generated variables ====
json_path = f"{lightning_log_dir}/{exp_name}.json"


## Mode: Hyperparameter Tuning ----
# loaded_json = False
# if os.path.exists(json_path): 
#     ax_client = (AxClient.load_from_json_file(filepath = json_path))
#     loaded_json = True

# else:
#     ax_client = AxClient()
#     ax_client.create_experiment(
#         name=exp_name,
#         parameters=params_list,
#         objectives={"train_loss": ObjectiveProperties(minimize=True)}
#     )

# run_trials_bool = True
# if run_hyps_force == False:
#     if loaded_json: 
#         # check if we've reached the max number of hyperparamters combinations to test
#         if max_hyps <= (ax_client.generation_strategy.trials_as_df.index.max()+1):
#             run_trials_bool = False

# if run_trials_bool:
#     # run the trials
#     for i in range(run_hyps):
#         parameterization, trial_index = ax_client.get_next_trial()
#         # Local evaluation here can be replaced with deployment to external system.
#         ax_client.complete_trial(trial_index=trial_index, raw_data=evaluate(parameterization))

#     ax_client.save_to_json_file(filepath = json_path)




## Mode: Training from ax_client ----
## Generated variables ====
# json_path = f"{lightning_log_dir}/{exp_name}.json"

# if os.path.exists(json_path): 
#     ax_client = (AxClient.load_from_json_file(filepath = json_path))
#     params, _ = ax_client.get_best_parameters()




## Mode: Training without ax_client ----

In [ ]:
ax_client.get_trials_data_frame()

In [ ]:
iter(training_dataloader)

In [ ]:
from vnnpaper.zma import plDNN_general

??plDNN_general

In [ ]:
# cxn
# sparsevnn.util.convert_connections(inp=cxn, to='dict', node_names=None)`)
# cxn_order_dict = sparsevnn.util.order_connections(inp = cxn, node_names=None)
# sum([cxn_order_dict[e] for e in list(cxn_order_dict.keys())], [])

In [ ]:
from vnnpaper.zma import \
    BigDataset,    \
    plDNN_general, \
    mask_parents,  \
    vnn_factory_1, \
    vnn_factory_2, \
    vnn_factory_3

# Hyperparameter Tuning ----
import os # needed for checking history (saved by lightning) 

## Logging with Pytorch Lightning ====
import lightning.pytorch as pl
from   lightning.pytorch.loggers import CSVLogger # used to save the history of each trial (used by ax)

## Adaptive Experimentation Platform ====
from ax.service.ax_client import AxClient, ObjectiveProperties